In [217]:
import sys
sys.path.append("..")

In [218]:
import itertools
import numpy as np
import pandas as pd
import plotly.express as px
from src.mip import solve

In [107]:
M = [11, 21]
N = [4, 5, 6]

df = pd.DataFrame()
for m in M:
    for n in N:
        df_mn = pd.read_pickle(f"grid_search_solver_m{m}_n{n}.pkl")
        df_mn["m"] = m
        df_mn["n"] = n
        df_gdy_mn = pd.read_pickle(f"grid_search_greedy_m{m}_n{n}.pkl")
        df_gdy_mn["m"] = m
        df_gdy_mn["n"] = n
        df = pd.concat((df, df_mn, df_gdy_mn))

df = df.sort_values(["m", "n", "i", "alg"]).reset_index(drop=True)
df.loc[df["alg"]=="pruned", "alg"] = "MILP_R"
df.loc[df["alg"]=="brute force", "alg"] = "OPT"
df.loc[df["alg"]=="agg", "alg"] = "AGG"
df.loc[df["alg"]=="div", "alg"] = "DIV"

In [251]:
df.head(10)

,i,alg,time,loss,partition,m,n
0,0,AGG,0.003695,0.081818,"[[0], [1], [2, 3]]",11,4
1,0,OPT,0.001121,0.081818,"[[0], [1], [2, 3]]",11,4
2,0,DIV,0.003769,0.081818,"[[0], [1], [2, 3]]",11,4
3,0,MILP_R,0.056114,0.086364,"[[0], [1, 2], [3]]",11,4
4,1,AGG,0.003029,0.081818,"[[0], [1], [2, 3]]",11,4
5,1,OPT,0.001274,0.081818,"[[0], [1], [2, 3]]",11,4
6,1,DIV,0.003335,0.081818,"[[0], [1], [2, 3]]",11,4
7,1,MILP_R,0.055469,0.086364,"[[0], [1], [2], [3]]",11,4
8,2,AGG,0.003077,0.086364,"[[0], [3], [1, 2]]",11,4
9,2,OPT,0.001049,0.086364,"[[0], [1, 2], [3]]",11,4


In [109]:
df[df['alg']=='MILP_R'].describe()

,i,time,loss,m,n
count,32946.000000,32946.000000,32946.000000,32946.000000,32946.000000
mean,4588.000000,0.438687,0.054720,16.000000,5.647059
std,3457.882819,0.438405,0.024289,5.000076,0.588244
min,0.000000,0.048071,0.004545,11.000000,4.000000
25%,1574.000000,0.153077,0.036364,11.000000,5.000000
50%,3633.000000,0.253930,0.059091,16.000000,6.000000
75%,7509.000000,0.551979,0.073810,21.000000,6.000000
max,11627.000000,2.402603,0.092857,21.000000,6.000000


In [110]:
res = {"m": [], "n": [], "matches": [], "total": [], "match %": []}
for m in M:
    for n in N:
        df_mn = df[(df["m"]==m) & (df["n"]==n)]
        matches = (df_mn[df_mn["alg"]=="MILP_R"]["loss"].reset_index(drop=True) - df_mn[df_mn["alg"]=="OPT"]["loss"].reset_index(drop=True)) < 1e-9
        res["m"].append(m)
        res["n"].append(n)
        res["matches"].append(matches.sum())
        res["total"].append(matches.count())
        res["match %"].append(matches.mean())

In [58]:
df_res = pd.DataFrame(res)
df_res

,m,n,matches,total,match %
0,11,4,962,969,0.992776
1,11,5,3875,3876,0.999742
2,11,6,11625,11628,0.999742
3,21,4,962,969,0.992776
4,21,5,3875,3876,0.999742
5,21,6,11572,11628,0.995184


In [ ]:
df_agg = df.groupby(["m", "n", "alg"], as_index=False).mean(True)
df_agg.head()

,m,n,alg,i,time,loss
0,11,4,brute force,484.0,0.001019,0.049906
1,11,4,pruned,484.0,0.060281,0.049948
2,11,5,brute force,1937.5,0.004032,0.047591
3,11,5,pruned,1937.5,0.101555,0.047592
4,11,6,brute force,5813.5,0.018862,0.044336


In [61]:
px.scatter(df_agg, x="n", y="time", color="alg", facet_col="m", labels={"time": "time (s)"})

In [188]:
res_ratio = {"m": [], "n": [], "i": [], "alg": [], "ratio": [], "time": []}
for m in M:
    for n in N:
        df_mn = df[(df["m"]==m) & (df["n"]==n)]
        for alg in ["OPT", "MILP_R"]:
            r = (df_mn[df_mn["alg"]==alg]["loss"].reset_index(drop=True) / df_mn[df_mn["alg"]=="OPT"]["loss"].reset_index(drop=True))
            t = df_mn[df_mn["alg"]==alg]["time"]
            i = df_mn[df_mn["alg"]==alg]["i"]

            res_ratio["m"].extend([m]*len(r))
            res_ratio["n"].extend([n]*len(r))
            res_ratio["i"].extend(i)
            res_ratio["alg"].extend([alg]*len(r))
            res_ratio["ratio"].extend(r.tolist())
            res_ratio["time"].extend(t.tolist())

        r_gdy = np.minimum(df_mn[df_mn["alg"]=="AGG"].loss.to_numpy(), df_mn[df_mn["alg"]=="DIV"].loss.to_numpy()) / df_mn[df_mn["alg"]=="OPT"]["loss"].reset_index(drop=True)
        t_gdy = (df_mn[df_mn["alg"]=="AGG"].time.to_numpy() + df_mn[df_mn["alg"]=="DIV"].time.to_numpy())
        i_gdy = df_mn[df_mn["alg"]=="AGG"]["i"]
        
        res_ratio["m"].extend([m]*len(r_gdy))
        res_ratio["n"].extend([n]*len(r_gdy))
        res_ratio["i"].extend(i_gdy)
        res_ratio["alg"].extend(["GDY"]*len(r_gdy))
        res_ratio["ratio"].extend(r_gdy.tolist())
        res_ratio["time"].extend(t_gdy.tolist())

In [189]:
df_res_ratio = pd.DataFrame(res_ratio)
df_res_ratio

,m,n,i,alg,ratio,time
0,11,4,0,OPT,1.0,0.001121
1,11,4,1,OPT,1.0,0.001274
2,11,4,2,OPT,1.0,0.001049
3,11,4,3,OPT,1.0,0.001027
4,11,4,4,OPT,1.0,0.001011
...,...,...,...,...,...,...
98833,21,6,11623,GDY,1.0,0.016382
98834,21,6,11624,GDY,1.0,0.012277
98835,21,6,11625,GDY,1.0,0.014302
98836,21,6,11626,GDY,1.0,0.011347


In [190]:
idx = pd.IndexSlice
df_res_ratio.groupby(["alg", "m", "n"]).describe().loc[:, idx[["ratio", "time"], ["mean", "std", "min", "max"]]]

ratio                                   time            \
                 mean           std  min       max      mean       std   
alg    m  n                                                              
GDY    11 4  1.000000  7.056038e-17  1.0  1.000000  0.006449  0.000771   
          5  1.000975  3.954559e-02  1.0  3.333333  0.012519  0.001826   
          6  1.015148  1.436403e-01  1.0  4.000000  0.021803  0.004221   
       21 4  1.000000  7.433912e-17  1.0  1.000000  0.006479  0.000817   
          5  1.000868  1.594689e-02  1.0  1.600000  0.012191  0.001912   
          6  1.009992  7.571816e-02  1.0  3.625000  0.019668  0.004070   
MILP_R 11 4  1.000631  8.174007e-03  1.0  1.166667  0.060281  0.013391   
          5  1.000017  1.070821e-03  1.0  1.066667  0.101555  0.027070   
          6  1.000055  3.897529e-03  1.0  1.375000  0.226075  0.122718   
       21 4  1.000296  3.716422e-03  1.0  1.062500  0.139488  0.034453   
          5  1.000007  4.589233e-04  1.0  1.028571  0.290511  0.121911   
          6  1.000678  1.195745e-02  1.0  1.500000  0.869537  0.478293   
OPT    11 4  1.000000  0.000000e+00  1.0  1.000000  0.001019  0.000042   
          5  1.000000  0.000000e+00  1.0  1.000000  0.004032  0.000340   
          6  1.000000  0.000000e+00  1.0  1.000000  0.018862  0.004415   
       21 4  1.000000  0.000000e+00  1.0  1.000000  0.001141  0.000128   
          5  1.000000  0.000000e+00  1.0  1.000000  0.004191  0.000273   
          6  1.000000  0.000000e+00  1.0  1.000000  0.018051  0.001144   

                                 
                  min       max  
alg    m  n                      
GDY    11 4  0.003770  0.011469  
          5  0.006232  0.027834  
          6  0.008983  0.047915  
       21 4  0.003829  0.014967  
          5  0.006074  0.027019  
          6  0.008922  0.053562  
MILP_R 11 4  0.048071  0.102799  
          5  0.066841  0.343291  
          6  0.090097  2.207894  
       21 4  0.100150  0.310969  
          5  0.146970  1.284809  
          6  0.205735  2.402603  
OPT    11 4  0.000915  0.001508  
          5  0.003718  0.008938  
          6  0.016475  0.101547  
       21 4  0.000997  0.002764  
          5  0.003819  0.008590  
          6  0.016832  0.062779

In [254]:
df_res_ratio_agg = df_res_ratio.groupby(["alg", "m", "n"], as_index=False).mean()
px.scatter(df_res_ratio_agg, x="m", y="time", color="alg", facet_col="n")

In [197]:
def generate_prior_grid(n_components, n_balls=50):
    step = 1.0 / n_balls
    grids = []
    for combo in itertools.combinations_with_replacement(range(n_components), n_balls - n_components):
        counts = np.bincount(combo, minlength=n_components) + 1
        grids.append((counts * step).round(4))
    return grids

In [ ]:
df_milp_over_1 = df_res_ratio[(df_res_ratio["alg"]=="MILP_R") & (df_res_ratio["ratio"]> 1+1e-12)]
print(f"{len(df_milp_over_1)} examples where solver and OPT mismatch")

75 examples where solver and OPT mismatch


m              21
n               6
i            5001
alg        MILP_R
ratio         1.5
time     2.210009
Name: 80583, dtype: object

In [252]:
df_milp_over_1

,m,n,i,alg,ratio,time
969,11,4,0,MILP_R,1.055556,0.056114
970,11,4,1,MILP_R,1.055556,0.055469
977,11,4,8,MILP_R,1.066667,0.052862
978,11,4,9,MILP_R,1.066667,0.050286
986,11,4,17,MILP_R,1.066667,0.055120
...,...,...,...,...,...,...
84684,21,6,9102,MILP_R,1.125000,2.209345
85966,21,6,10384,MILP_R,1.250000,0.615703
86455,21,6,10873,MILP_R,1.400000,2.140914
86559,21,6,10977,MILP_R,1.066667,1.053373


In [233]:
row = df_milp_over_1.iloc[df_milp_over_1["ratio"].argmax()]
m, n, i = row.m, row.n, row.i
df[(df["m"]==m) & (df["n"]==n) & (df["i"]==i)]

,i,alg,time,loss,partition,m,n
105276,5001,AGG,0.008428,0.042857,"[[1], [5], [0, 2, 3, 4]]",21,6
105277,5001,OPT,0.017796,0.042857,"[[1], [0, 2, 3, 4], [5]]",21,6
105278,5001,DIV,0.008992,0.042857,"[[0, 2, 3, 4], [5], [1]]",21,6
105279,5001,MILP_R,2.210009,0.064286,"[[0, 1], [2, 3, 4, 5]]",21,6


In [234]:
pg = generate_prior_grid(n, 20)
priors = pg[i]
thresholds = np.linspace(0., 1., n)
tt = 0.1
c = 1.0

mip = solve(priors, thresholds, tt, c, m)
mip["loss"], mip["partition"]

(0.04285714285714286, [[0, 2, 3, 4], [1], [5]])

In [237]:
df[(df["m"]==m) & (df["n"]==n) & (df["i"]==i)].iloc[1].loss

np.float64(0.04285714285714286)

In [242]:
results_over_1 = {"i": [], "loss": [], "ratio": [], "partition": []}
for _, row in df_milp_over_1.iterrows():
    m, n, i = row.m, row.n, row.i
    pg = generate_prior_grid(n, 20)
    priors = pg[i]
    thresholds = np.linspace(0., 1., n)
    tt = 0.1
    c = 1.0

    mip = solve(priors, thresholds, tt, c, m)
    p_mip, loss_mip = mip["partition"], mip["loss"]
    r_mip = loss_mip / df[(df["m"]==m) & (df["n"]==n) & (df["i"]==i)].iloc[1].loss
    
    results_over_1["i"].append(i)
    results_over_1["loss"].append(loss_mip)
    results_over_1["ratio"].append(r_mip)
    results_over_1["partition"].append(p_mip)

In [249]:
df_results_over_1 = pd.DataFrame(results_over_1)
df_fails = df_results_over_1[df_results_over_1["ratio"] > 1+1e-12]
print(f"Even after removing time limit {len(df_fails)} examples weren't solved")

Even after removing time limit 34 examples weren't solved


In [253]:
df_fails

,i,loss,ratio,partition
0,0,0.086364,1.055556,"[[0], [1, 2], [3]]"
1,1,0.086364,1.055556,"[[0], [1], [2], [3]]"
2,8,0.072727,1.066667,"[[0], [1], [2, 3]]"
3,9,0.072727,1.066667,"[[0], [1, 3], [2]]"
4,17,0.072727,1.066667,"[[0, 1], [2, 3]]"
5,27,0.077273,1.133333,"[[0, 1], [2, 3]]"
6,48,0.063636,1.166667,"[[0], [1], [2, 3]]"
7,4,0.072727,1.066667,"[[0, 1], [2, 3, 4]]"
8,4176,0.050000,1.375000,"[[0, 1], [2, 4, 5], [3]]"
9,5061,0.036364,1.142857,"[[0, 2], [1], [3, 4, 5]]"
